# AI of GOD 5 — Swin Transformer V2 | 5-Fold CV | Macro F1
- **Model:** `swinv2_base_window12to16_192to256` via timm (fits 15 GB T4)
- **Input:** 256×256
- **CV:** Stratified 5-Fold, checkpoint by Val Macro F1
- **Inference:** 5-fold ensemble + 2× TTA

In [1]:
import os, gc, zipfile, warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
CFG = dict(
    # swinv2_base_window12to16_192to256  → safe on 15 GB, ImageNet-22k pretrained
    # swap to swinv2_large_window12to16_192to256 if you have >24 GB
    model_name   = 'swinv2_base_window12to16_192to256.ms_in22k_ft_in1k',
    img_size     = 256,
    n_folds      = 5,
    seed         = 42,
    epochs       = 12,
    batch_size   = 16,
    grad_accum   = 2,
    backbone_lr  = 2e-5,
    head_lr      = 2e-4,
    weight_decay = 1e-4,
    warmup_frac  = 0.1,
    label_smooth = 0.1,
    mixed_prec   = True,
    num_workers  = 4,
    output_dir   = '/kaggle/working',
)
os.makedirs(CFG['output_dir'], exist_ok=True)

def seed_everything(s):
    import random; random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['seed'])

In [3]:
# ─── Data ─────────────────────────────────────────────────────────────────────
def find_file(name):
    for r, _, files in os.walk('/kaggle/input'):
        if name in files: return os.path.join(r, name)
    raise FileNotFoundError(name)

train_df = pd.read_csv(find_file('train.csv'))
sub_df   = pd.read_csv(find_file('sample_submission.csv'))
base     = os.path.dirname(find_file('train.csv'))
TRAIN_DIR, TEST_DIR = base+'/train_images', base+'/test_images'

classes           = sorted(train_df['animal_class'].unique())
class_to_idx      = {c: i for i, c in enumerate(classes)}
idx_to_class      = {i: c for c, i in class_to_idx.items()}
train_df['label'] = train_df['animal_class'].map(class_to_idx)
CFG['num_classes'] = len(classes)
print(f'Train {len(train_df)} | Test {len(sub_df)} | Classes {CFG["num_classes"]}')

Train 3087 | Test 1179 | Classes 72


In [4]:
# ─── Transforms ───────────────────────────────────────────────────────────────
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG['img_size']

train_tfm = T.Compose([
    T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomCrop(S),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.RandomRotation(15),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
val_tfm = T.Compose([
    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
tta_tfm = T.Compose([
    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

In [5]:
# ─── Dataset ──────────────────────────────────────────────────────────────────
class AnimalDataset(Dataset):
    def __init__(self, df, image_dir, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, row['image_id'])).convert('RGB')
        img = self.transform(img)
        if self.is_test: return img, row['image_id']
        return img, torch.tensor(row['label'], dtype=torch.long)

In [6]:
# ─── Model ────────────────────────────────────────────────────────────────────
def build_model():
    m = timm.create_model(
        CFG['model_name'],
        pretrained=True,
        num_classes=CFG['num_classes'],
        img_size=CFG['img_size'],
    )
    return m.to(device)

m = build_model()
print(f'Params: {sum(p.numel() for p in m.parameters())/1e6:.1f}M')
del m; gc.collect(); torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  357MB            

model.safetensors: downloading bytes:           |  0.00B            

Params: 87.0M


In [7]:
# ─── Optimizer / scheduler / loss ─────────────────────────────────────────────
def make_opt_sched(model, total_steps):
    head  = list(model.head.parameters())
    head_ids = {id(p) for p in head}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([
        {'params': backbone, 'lr': CFG['backbone_lr']},
        {'params': head,     'lr': CFG['head_lr']},
    ], weight_decay=CFG['weight_decay'])
    warmup = int(total_steps * CFG['warmup_frac'])
    def lr_fn(step):
        if step < warmup: return step / max(1, warmup)
        p = (step - warmup) / max(1, total_steps - warmup)
        return max(0.0, 0.5 * (1 + np.cos(np.pi * p)))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])

@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, labels, loss = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
            logits = model(x)
        loss += criterion(logits, y).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return loss/len(loader), f1_score(labels, preds, average='macro', zero_division=0)

In [8]:
# ─── 5-Fold Training ──────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
oof_probs   = np.zeros((len(train_df), CFG['num_classes']))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label'])):
    print(f'\n{"="*55}\n  FOLD {fold+1}/{CFG["n_folds"]}\n{"="*55}')
    seed_everything(CFG['seed'] + fold)

    tr_dl = DataLoader(AnimalDataset(train_df.iloc[tr_idx], TRAIN_DIR, train_tfm),
                       batch_size=CFG['batch_size'], shuffle=True,
                       num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    va_dl = DataLoader(AnimalDataset(train_df.iloc[va_idx], TRAIN_DIR, val_tfm),
                       batch_size=CFG['batch_size'], shuffle=False,
                       num_workers=CFG['num_workers'], pin_memory=True)

    model   = build_model()
    total_steps = (len(tr_dl) // CFG['grad_accum']) * CFG['epochs']
    opt, sched  = make_opt_sched(model, total_steps)
    scaler      = torch.cuda.amp.GradScaler(enabled=CFG['mixed_prec'])
    best_f1     = 0.0
    ckpt        = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')

    for epoch in range(CFG['epochs']):
        model.train(); opt.zero_grad(); tr_loss = 0.0
        for step, (x, y) in enumerate(tqdm(tr_dl,
                desc=f'  Ep {epoch+1:2d}/{CFG["epochs"]}', leave=False)):
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
                loss = criterion(model(x), y) / CFG['grad_accum']
            scaler.scale(loss).backward()
            tr_loss += loss.item() * CFG['grad_accum']
            if (step+1) % CFG['grad_accum'] == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()

        vl, vf1 = validate(model, va_dl)
        flag = '  ✓' if vf1 > best_f1 else ''
        print(f'  Ep {epoch+1:2d} | tr {tr_loss/len(tr_dl):.4f} '
              f'| val {vl:.4f} | F1 {vf1:.4f}{flag}')
        if vf1 > best_f1:
            best_f1 = vf1
            torch.save({'model_state': model.state_dict(), 'fold': fold+1,
                        'epoch': epoch+1, 'val_macro_f1': best_f1,
                        'classes': classes, 'cfg': CFG}, ckpt)

    print(f'  → best F1: {best_f1:.4f}')
    fold_scores.append(best_f1)

    # OOF
    model.load_state_dict(torch.load(ckpt)['model_state'])
    model.eval(); offset = 0
    with torch.no_grad():
        for x, _ in DataLoader(AnimalDataset(train_df.iloc[va_idx], TRAIN_DIR, val_tfm),
                                batch_size=CFG['batch_size'],
                                num_workers=CFG['num_workers']):
            probs = F.softmax(model(x.to(device)), 1).cpu().numpy()
            oof_probs[va_idx[offset:offset+len(probs)]] = probs
            offset += len(probs)

    del model, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()

cv_mean, cv_std = np.mean(fold_scores), np.std(fold_scores)
oof_f1 = f1_score(train_df['label'], oof_probs.argmax(1), average='macro', zero_division=0)
print(f'\nCV: {cv_mean:.4f} ± {cv_std:.4f}  |  OOF F1: {oof_f1:.4f}')


  FOLD 1/5


  Ep  1/12:   0%|          | 0/154 [00:00<?, ?it/s]

  Ep  1 | tr 4.1002 | val 3.1097 | F1 0.5070  ✓


  Ep  2/12:   0%|          | 0/154 [00:00<?, ?it/s]

  Ep  2 | tr 1.5634 | val 1.0323 | F1 0.9361  ✓


...

  Ep 10 | tr 0.7665 | val 0.8227 | F1 0.9817


  Ep 11/12:   0%|          | 0/154 [00:00<?, ?it/s]

  Ep 11 | tr 0.7650 | val 0.8219 | F1 0.9835


  Ep 12/12:   0%|          | 0/154 [00:00<?, ?it/s]

  Ep 12 | tr 0.7647 | val 0.8218 | F1 0.9835
  → best F1: 0.9869

CV: 0.9834 ± 0.0022  |  OOF F1: 0.9834


In [9]:
# ─── Inference: 5-fold × 2 TTA ────────────────────────────────────────────────
test_probs = np.zeros((len(sub_df), CFG['num_classes']))

for fold in range(CFG['n_folds']):
    model = build_model()
    model.load_state_dict(
        torch.load(os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt'))['model_state'])
    model.eval()
    fold_p = np.zeros_like(test_probs)

    for tfm in [val_tfm, tta_tfm]:
        dl = DataLoader(AnimalDataset(sub_df, TEST_DIR, tfm, is_test=True),
                        batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'])
        buf = []
        with torch.no_grad():
            for x, _ in tqdm(dl, desc=f'Fold {fold+1}', leave=False):
                with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
                    buf.append(F.softmax(model(x.to(device)), 1).cpu().numpy())
        fold_p += np.vstack(buf)

    test_probs += fold_p / 2
    del model; gc.collect(); torch.cuda.empty_cache()

test_probs /= CFG['n_folds']
sub_df['animal_class'] = [idx_to_class[i] for i in test_probs.argmax(1)]
sub_path = os.path.join(CFG['output_dir'], 'submission.csv')
sub_df[['image_id', 'animal_class']].to_csv(sub_path, index=False)
print('submission.csv saved')
print(sub_df.head())

Fold 1:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 1:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 2:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 2:   0%|          | 0/74 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fa1fffdefc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fa1fffdefc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

...

Fold 4:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 4:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 5:   0%|          | 0/74 [00:00<?, ?it/s]

Fold 5:   0%|          | 0/74 [00:00<?, ?it/s]

submission.csv saved
                               image_id animal_class
0          6eab26f8b156fda90e1bc819.png   woodpecker
1  483f71af1c174ab2a0c31a836267227d.png     antelope
2  563878772a3f47ce98cf9c10039a0219.png   chimpanzee
3  9aa782974228482390c1a237408a967a.png      hamster
4  3ed553219cf049b8bd9a65c300b02fb0.png          cat


In [10]:
# ─── Zip weights ──────────────────────────────────────────────────────────────
zip_path = os.path.join(CFG['output_dir'], 'swin_weights_all_folds.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fold in range(CFG['n_folds']):
        pt = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')
        zf.write(pt, arcname=f'fold{fold+1}_best.pt')
        print(f'  + fold{fold+1}_best.pt  ({os.path.getsize(pt)/1e6:.0f} MB)')
    zf.write(sub_path, arcname='submission.csv')
print(f'\n✅ Done  |  CV {cv_mean:.4f} ± {cv_std:.4f}  |  OOF {oof_f1:.4f}')

  + fold1_best.pt  (348 MB)
  + fold2_best.pt  (348 MB)
  + fold3_best.pt  (348 MB)
  + fold4_best.pt  (348 MB)
  + fold5_best.pt  (348 MB)

✅ Done  |  CV 0.9834 ± 0.0022  |  OOF 0.9834
